# Autonomous Master Manuscript Processing & Packaging Pipeline
### Turnkey One-Click Batch Pipeline: Processes All Books from `pdf/` and Generates Submission Deliverables in `output/`
**Course**: CSC15006 - Introduction to Natural Language Processing (NLP)  
**Submission Specification**: `guide/Quy_định_đồ_án.pdf` (FIT HCMUS)  

---

## 1. Overview

This turnkey notebook operates autonomously with a **single click ("Restart & Run All")**:
1. **Input**: Reads all book files directly from the manuscript folder `datasets-midterm/pdf/` (23 scanned PDFs + 1 companion translation file `Thụ Dạ Lữ Hoài Ngâm Khúc_Dịch.txt`).
2. **Processing**: High-fidelity 300 DPI rendering, layout classification, 100% page conservation across `ch/`, `vi/`, and `misc/`, bilingual alignment via Han-Nom phonetic dictionary, line segmentation, and PPOCRLabel 4-point polygon bounding box generation.
3. **Deliverables in `output/<Tên sách>/`**:
   - `<Tên sách>.pdf`: Replicated original source PDF.
   - `ch/`: High-resolution scans + `Label.txt` (4-point polygon boxes) + `fileState.txt`.
   - `vi/`: High-resolution scans + `Label.txt` (4-point polygon boxes) + `fileState.txt`.
   - `misc/`: Cover, index, and bibliographic pages (100% page conservation).
   - `<Tên sách>.csv`: UTF-8 with BOM bilingual parallel corpus (4 standard columns).
   - `<Tên sách>.xlsx`: Styled Excel workbook with ARGB font colors (Black `FF000000`, Red `FFFF0000`).
4. **Compliance Audit**: Automatically verifies all 10 regulatory criteria and outputs a unified scorecard.

**Quick Start**: Click **Kernel -> Restart Kernel and Run All Cells** to generate the complete `output/` folder.


In [ ]:
# Environment Setup and Core Module Imports
import os
import sys
import re
import time
import json
import shutil
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import cv2
import numpy as np
import pandas as pd
import openpyxl
import pymupdf

# Set UTF-8 encoding for standard output streams on Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import (
    BASE_DIR, OUTPUT_DIR, DATASETS_MIDTERM_DIR, MIN_SIMILARITY_THRESHOLD,
    COLOR_BLACK, COLOR_RED, KIMHANNOM_BASE_URL
)
from core.classifier import BookClassifier
from core.preprocess import DocumentPreprocessor, cv2_imread_unicode
from core.cleaner import TextCleaner, PPOCRLabelFormatter
from core.dictionary import HanNomDictionary
from core.aligner import SentenceAligner
from core.evaluator import AlignmentEvaluator
from core.exporter import CorpusExporter

PDF_DIR = DATASETS_MIDTERM_DIR / "pdf"
CSV_DIR = DATASETS_MIDTERM_DIR / "csv"
XLSX_DIR = DATASETS_MIDTERM_DIR / "xlsx"

lexicon_size = len(HanNomDictionary.get_instance().char_to_readings)

print("=" * 80)
print("AUTONOMOUS MASTER PIPELINE INITIALIZED")
print("=" * 80)
print(f"Manuscript PDF Directory : {PDF_DIR}")
print(f"Deliverables Output Dir  : {OUTPUT_DIR}")
print(f"Phonetic Lexicon Entries : {lexicon_size} loaded")
print("Core modules loaded successfully.")
print("=" * 80)


In [ ]:
# Scan PDF Folder and Discover All Manuscript Entities
classifier = BookClassifier()
all_pdf_files = sorted(list(PDF_DIR.glob("*.pdf")))
txt_companion = PDF_DIR / "Thụ Dạ Lữ Hoài Ngâm Khúc_Dịch.txt"

catalog = []
for idx, pdf_path in enumerate(all_pdf_files, 1):
    stem = pdf_path.stem
    doc = pymupdf.open(str(pdf_path))
    pcount = len(doc)
    doc.close()
    
    csv_path = CSV_DIR / f"{stem}.csv"
    xlsx_path = XLSX_DIR / f"{stem}.xlsx"
    cls_res = classifier.classify(pdf_path)
    
    catalog.append({
        "STT": idx,
        "Book_Key": stem,
        "PDF_Filename": pdf_path.name,
        "Total_Pages": pcount,
        "Layout_Code": cls_res["book_type"],
        "Layout_Name": cls_res["type_name"],
        "Confidence": f"{cls_res['confidence'] * 100:.0f}%",
        "PDF_Path": pdf_path,
        "CSV_Path": csv_path if csv_path.exists() else None,
        "XLSX_Path": xlsx_path if xlsx_path.exists() else None,
    })

print("=" * 95)
print(f"DISCOVERED {len(catalog)} MANUSCRIPT ENTITIES IN {PDF_DIR.name}/")
if txt_companion.exists():
    print(f"Companion Translation Detected: {txt_companion.name}")
print("=" * 95)
print(f"{'STT':<4} | {'Code':<6} | {'Pages':<5} | {'Conf':<6} | {'Book Title':<50}")
print("=" * 95)
total_all_pages = sum(b['Total_Pages'] for b in catalog)
for b in catalog:
    print(f"{b['STT']:<4} | {b['Layout_Code']:<6} | {b['Total_Pages']:<5} | {b['Confidence']:<6} | {b['Book_Key'][:50]:<50}")
print("=" * 95)
print(f"TOTAL MANUSCRIPTS: {len(catalog)} | TOTAL PAGES ACROSS CATALOG: {total_all_pages}")
print("=" * 95)


In [ ]:
# Master Pipeline Execution Engine for All Books
t_start_all = time.time()
execution_results = []

print("=" * 80)
print(f"STARTING PIPELINE EXECUTION FOR ALL {len(catalog)} MANUSCRIPTS")
print("=" * 80)

for b_idx, book_info in enumerate(catalog, 1):
    stem = book_info["Book_Key"]
    pdf_path = book_info["PDF_Path"]
    csv_path = book_info["CSV_Path"]
    layout_code = book_info["Layout_Code"]
    t0 = time.time()
    
    print(f"\n[{b_idx:02d}/{len(catalog):02d}] Processing: {stem}")
    print(f"  Layout: {layout_code} | Total Pages: {book_info['Total_Pages']}")
    
    b_dict = {
        "title": stem,
        "pdf_filename": pdf_path.name,
        "csv_filename": csv_path.name if csv_path else f"{stem}.csv",
        "layout_type": layout_code
    }
    
    # Step 1: Preprocessing & Rendering 300 DPI Images
    prep = DocumentPreprocessor(b_dict, OUTPUT_DIR)
    book_out_dir = OUTPUT_DIR / stem
    book_out_dir.mkdir(parents=True, exist_ok=True)
    extracted_pages = prep.split_and_extract_images(str(pdf_path))
    
    c_pages, misc_pages, blank_pages = [], [], []
    for pno, name, img_p in extracted_pages:
        cat = prep.classify_page(img_p, pno, len(extracted_pages))
        if cat == "content":
            c_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(prep.dir_ch / name))
            shutil.copy2(str(img_p), str(prep.dir_vi / name))
        elif cat == "blank":
            blank_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(prep.dir_blank / name))
        else:
            misc_pages.append((pno, name, img_p))
            shutil.copy2(str(img_p), str(prep.dir_misc / name))
            
    # Step 2: Replicate Source PDF to Deliverables Folder
    dest_pdf = book_out_dir / f"{stem}.pdf"
    shutil.copy2(str(pdf_path), str(dest_pdf))
    
    # Step 3: Sentence Alignment & Reconstruction
    aligner = SentenceAligner(b_dict, csv_path) if csv_path else None
    recs = aligner.align_book_data() if aligner else []
    
    # Step 4: Segment Bounding Polygons & Generate PPOCRLabel Annotations
    page_map = {}
    for r in recs:
        img_id = r["img_id"]
        if img_id not in page_map:
            page_map[img_id] = []
        page_map[img_id].append(r)
        
    ch_map, vi_map = {}, {}
    for pno, img_name, img_p in c_pages:
        seg = prep.segment_t2b_lines(img_p)
        page_recs = page_map.get(img_name, [])
        ch_boxes = seg["ch"]
        vi_boxes = seg["vi"]
        
        ch_map[img_name] = [
            {"transcription": page_recs[i]["SinoNom_Char"] if i < len(page_recs) else "", "points": b["points"]}
            for i, b in enumerate(ch_boxes)
        ]
        vi_map[img_name] = [
            {"transcription": page_recs[i]["ChuQN_txt"] if i < len(page_recs) else "", "points": b["points"]}
            for i, b in enumerate(vi_boxes)
        ]
        
    PPOCRLabelFormatter.write_ppocr_dataset(prep.dir_ch, "ch", ch_map)
    PPOCRLabelFormatter.write_ppocr_dataset(prep.dir_vi, "vi", vi_map)
    
    # Step 5: Quality & Similarity Evaluation
    evaluator = AlignmentEvaluator()
    eval_rep = evaluator.evaluate_dataset(recs)
    
    # Step 6: Export CSV (UTF-8 BOM) and Styled XLSX (ARGB Colors)
    out_csv = book_out_dir / f"{stem}.csv"
    out_xlsx = book_out_dir / f"{stem}.xlsx"
    CorpusExporter.export_csv(recs, out_csv)
    CorpusExporter.export_xlsx(recs, eval_rep["row_evaluations"], out_xlsx)
    
    elapsed = time.time() - t0
    print(f"  Completed in {elapsed:.2f}s | Aligned Pairs: {len(recs)} | Similarity: {eval_rep['similarity_percentage']:.2f}%")
    
    execution_results.append({
        "book": stem,
        "layout": layout_code,
        "total_pages": len(extracted_pages),
        "content_pages": len(c_pages),
        "misc_pages": len(misc_pages),
        "blank_pages": len(blank_pages),
        "aligned_pairs": len(recs),
        "total_chars": eval_rep["total_chars"],
        "similarity": eval_rep["similarity_percentage"],
        "passed": eval_rep["is_passed"],
        "elapsed_sec": round(elapsed, 2),
        "out_dir": book_out_dir,
        "pdf_path": dest_pdf,
        "csv_path": out_csv,
        "xlsx_path": out_xlsx,
        "ch_dir": prep.dir_ch,
        "vi_dir": prep.dir_vi,
        "misc_dir": prep.dir_misc
    })

total_elapsed = time.time() - t_start_all
print("\n" + "=" * 80)
print(f"ALL {len(catalog)} BOOKS PROCESSED SUCCESSFULLY IN {total_elapsed:.2f} SECONDS ({total_elapsed/60:.1f} MINUTES)")
print("=" * 80)


In [ ]:
# Automated 10/10 Compliance Verification for All Deliverable Folders
def validate_ppocr_format(label_path: Path) -> bool:
    if not label_path.exists():
        return False
    lines = [l.strip() for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not lines:
        return False
    for line in lines:
        parts = line.split("\t")
        if len(parts) != 2:
            return False
        try:
            boxes = json.loads(parts[1])
            for b in boxes:
                if len(b.get("points", [])) != 4:
                    return False
        except Exception:
            return False
    return True

audit_results = []
print("=" * 80)
print("AUDITING DELIVERABLES COMPLIANCE (10/10 CRITERIA) ACROSS ALL BOOKS")
print("=" * 80)

for res in execution_results:
    stem = res["book"]
    out_dir = res["out_dir"]
    ch_dir = res["ch_dir"]
    vi_dir = res["vi_dir"]
    misc_dir = res["misc_dir"]
    csv_file = res["csv_path"]
    xlsx_file = res["xlsx_path"]
    pdf_file = res["pdf_path"]
    
    checks = {}
    
    # 1. 100% Page conservation
    ch_count = len(list(ch_dir.glob("*.jpg")))
    misc_count = len(list(misc_dir.glob("*.jpg")))
    blank_count = len(list((out_dir / "blank").glob("*.jpg"))) if (out_dir / "blank").exists() else 0
    checks["1_Page_Conservation"] = (ch_count + misc_count + blank_count) >= res["total_pages"]
    
    # 2. Subdirectory ch/
    checks["2_Subdir_ch"] = (ch_dir / "Label.txt").exists() and (ch_dir / "fileState.txt").exists()
    
    # 3. Subdirectory vi/
    checks["3_Subdir_vi"] = (vi_dir / "Label.txt").exists() and (vi_dir / "fileState.txt").exists()
    
    # 4. Subdirectory misc/
    checks["4_Subdir_misc"] = misc_dir.is_dir() and len(list(misc_dir.glob("*.jpg"))) > 0
    
    # 5. Original PDF replicated
    checks["5_Replicated_PDF"] = pdf_file.exists() and pdf_file.stat().st_size > 0
    
    # 6. ch/Label.txt format
    checks["6_ch_PPOCRLabel_4pt"] = validate_ppocr_format(ch_dir / "Label.txt")
    
    # 7. vi/Label.txt format
    checks["7_vi_PPOCRLabel_4pt"] = validate_ppocr_format(vi_dir / "Label.txt")
    
    # 8. CSV UTF-8 with BOM format
    try:
        df_csv = pd.read_csv(csv_file, encoding="utf-8-sig")
        checks["8_CSV_BOM_4Cols"] = list(df_csv.columns[:4]) == ["img_id", "sentence_id", "SinoNom_Char", "ChuQN_txt"]
    except Exception:
        checks["8_CSV_BOM_4Cols"] = False
        
    # 9. XLSX ARGB font styling
    try:
        wb = openpyxl.load_workbook(xlsx_file)
        ws = wb.active
        checks["9_XLSX_ARGB_Colors"] = any(c.font and c.font.color and c.font.color.rgb for row in ws.iter_rows(min_row=2, max_row=10) for c in row)
    except Exception:
        checks["9_XLSX_ARGB_Colors"] = False
        
    # 10. Similarity metric >= 95.0%
    checks["10_Phonetic_Sim_95"] = res["similarity"] >= 95.0
    
    pass_count = sum(1 for v in checks.values() if v)
    all_ok = pass_count == 10
    
    audit_results.append({
        "Book_Title": stem,
        "Total_Pages": res["total_pages"],
        "Content_Pages": res["content_pages"],
        "Misc_Pages": res["misc_pages"],
        "Aligned_Pairs": res["aligned_pairs"],
        "Similarity_%": f"{res['similarity']:.2f}%",
        "Audit_Score": f"{pass_count}/10",
        "Compliance": "PASS [10/10]" if all_ok else f"FAIL [{pass_count}/10]",
        "Runtime_s": f"{res['elapsed_sec']:.1f}s"
    })
    
    print(f"Book: {stem[:42]:<42} | Audit: {pass_count}/10 | Sim: {res['similarity']:.2f}% | {'PASS' if all_ok else 'FAIL'}")

print("=" * 80)

# Final Deliverables Audit Scorecard
scorecard_df = pd.DataFrame(audit_results)
print("\n" + "=" * 105)
print("FINAL DELIVERABLES AUDIT SCORECARD")
print("=" * 105)
print(scorecard_df.to_string(index=False))
print("=" * 105)

all_passed = all(r["Compliance"] == "PASS [10/10]" for r in audit_results)
if all_passed:
    print("\nOVERALL SYSTEM STATUS: ALL MANUSCRIPTS PASSED 10/10 COMPLIANCE AUDIT")
    print(f"Deliverables folder is fully verified and ready for submission: {OUTPUT_DIR}")
else:
    failed_books = [r['Book_Title'] for r in audit_results if r['Compliance'] != 'PASS [10/10]']
    print(f"\nWARNING: {len(failed_books)} book(s) require attention: {failed_books}")


## 2. Deliverables Structure Verification

According to **`guide/Quy_định_đồ_án.pdf` (Page 4)**, the output folder is structured as follows:

```text
output/
|-- <Tên sách>/
|   |-- ch/
|   |   |-- <Tên sách>.<Số thứ tự hình ảnh>.jpg
|   |   |-- Label.txt        (PPOCRLabel 4-point polygon format)
|   |   `-- fileState.txt    (PPOCRLabel status 1/0)
|   |-- vi/
|   |   |-- <Tên sách>.<Số thứ tự hình ảnh>.jpg
|   |   |-- Label.txt        (PPOCRLabel 4-point polygon format)
|   |   `-- fileState.txt    (PPOCRLabel status 1/0)
|   |-- misc/
|   |   `-- <Tên sách>.<Số thứ tự hình ảnh>.jpg (Covers, table of contents, dividers)
|   |-- <Tên sách>.pdf       (Original PDF replicated)
|   |-- <Tên sách>.xlsx      (Bilingual parallel corpus, ARGB Black/Red styled fonts)
|   `-- <Tên sách>.csv       (Bilingual parallel corpus, UTF-8 with BOM)
```

> [!NOTE]
> To submit, compress the target book folders in `output/` into a standard `.zip` archive as specified by course instructors.
